<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_2_DataFrame_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.2 — DataFrame Basics

## Learning Objectives

By the end of this notebook you should be able to:

- Explain a DataFrame as a collection of aligned Series
- Create a DataFrame from a dictionary and read one from a CSV file
- Inspect a DataFrame with `.head()`, `.tail()`, `.shape`, `.dtypes`, `.info()`, `.describe()`
- Select one column (returns a Series) and multiple columns (returns a DataFrame)
- Add a new computed column
- Drop rows or columns with `.drop()`

In [ ]:
import pandas as pd

## 1. What Is a DataFrame?

A **DataFrame** is a two-dimensional table — rows and columns — where every column is a `Series` sharing the same index. You can think of it as:
- A spreadsheet with labeled rows and columns.
- A dictionary mapping column names to Series objects.

```
         survived  pclass     sex   age     fare
index
0               0       3    male  22.0    7.25
1               1       1  female  38.0   71.28
2               1       3  female  26.0    7.92
```

Every column (`survived`, `pclass`, `sex`, …) is a `Series`. They all share the same row index (0, 1, 2, …).

In [ ]:
# Create a DataFrame from a dictionary
# Keys become column names; values (lists) become column data
data = {
    "name":     ["Alice", "Bob", "Carol", "Dan"],
    "age":      [22, 38, 26, 35],
    "survived": [1, 0, 1, 0],
    "fare":     [7.25, 71.28, 7.92, 53.10],
}

df_small = pd.DataFrame(data)
df_small

Each column really is a Series:

In [ ]:
print(type(df_small["age"]))   # <class 'pandas.core.series.Series'>
print(df_small["age"])

In [ ]:
# Another common pattern: a list of dicts, one dict per row
rows = [
    {"name": "Alice", "age": 22, "fare": 7.25},
    {"name": "Bob",   "age": 38, "fare": 71.28},
]
pd.DataFrame(rows)

## 2. Loading Data from a CSV

In practice you will almost always load data from a file or URL using `pd.read_csv()`.

In [ ]:
url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

## 3. Inspecting a DataFrame

Always inspect your data before doing any analysis. pandas provides several tools for this.

In [ ]:
# First 5 and last 5 rows
print("--- head ---")
print(df.head())
print()
print("--- tail ---")
print(df.tail())

In [ ]:
# .shape returns (rows, columns)
rows, cols = df.shape
print(f"The dataset has {rows} rows and {cols} columns.")

In [ ]:
# .dtypes — data type of each column
df.dtypes

`.info()` is one of the most useful first calls. It shows column names, data types, and non-null counts simultaneously. If the non-null count for any column is less than the total row count, that column has missing values.

In [ ]:
df.info()

In [ ]:
# .describe() — summary statistics for every numeric column
df.describe()

## 4. Selecting Columns

### Single column → Series

Use `df["column_name"]`. The result is a `Series`.

In [ ]:
age_col = df["age"]
print(type(age_col))
age_col.head()

### Multiple columns → DataFrame

Pass a **list** of column names. The result is a smaller DataFrame.

Notice the double brackets: the outer `[]` is the column-selection operator; the inner `[]` is a Python list of names.

In [ ]:
subset = df[["name", "sex", "age", "survived"]]
subset.head()

## 5. Adding a New Column

Assign directly to a new column name. The right-hand side can be a constant, a Series, or a vectorized expression.

In [ ]:
# Convert fare from British pounds to US dollars (approximate)
df["fare_usd"] = df["fare"] * 1.27

df[["name", "fare", "fare_usd"]].head()

In [ ]:
# Boolean column: did this passenger travel with family?
df["has_family"] = (df["sibsp"] > 0) | (df["parch"] > 0)

df[["name", "sibsp", "parch", "has_family"]].head(10)

## 6. Dropping Rows and Columns

`.drop()` returns a **new** DataFrame without the specified rows or columns.

- To drop **columns**: `df.drop(columns=[...])` or pass `axis=1`
- To drop **rows**: pass the row index label(s) with `axis=0` (default)

In [ ]:
# Drop the columns we just added (clean up)
df_trimmed = df.drop(columns=["fare_usd", "has_family"])
print("Columns:", df_trimmed.columns.tolist())

In [ ]:
# Drop row 0 (the first passenger)
df_no_first = df.drop(index=0)
print("Original rows:", len(df))
print("After drop:   ", len(df_no_first))

## 7. A Quick Look at the Titanic Data

Now that you know the basic inspection tools, let's ask a few quick questions about this dataset.

In [ ]:
# How many passengers survived?
print("Survived (1) vs Did not (0):")
print(df["survived"].value_counts())

In [ ]:
# Passenger class distribution
print("Class distribution:")
print(df["pclass"].value_counts().sort_index())

In [ ]:
# Age range
print("Age statistics:")
print(df["age"].describe().round(1))

In [ ]:
# Sex distribution
print("Sex distribution:")
print(df["sex"].value_counts())

## Your Turn

1. How many passengers have `sibsp > 0` (traveled with at least one sibling or spouse)?
2. What is the average fare paid?
3. Create a new column called `fare_per_class` defined as `fare / pclass`. Display the first 5 rows showing `name`, `fare`, `pclass`, and `fare_per_class`.
4. Drop the `name` column and confirm the resulting DataFrame has 7 columns.

In [ ]:
# Your code here